# B2.6 · Sandbox replication

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.5 · Feasibility filtering and reachability](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**.

| | |
|---|---|
| Tools used | Docker, gVisor, Cilium, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Stand up an isolated replica, prove egress and credential isolation, and show what a destructive probe touches.

**Why a security engineer needs it.** Dynamic testing is run against staging, so a destructive probe becomes an incident. The control it builds is: stage 11: replicate the application in an isolated, disposable runtime with no path to production.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You cannot exploit a finding to confirm it without somewhere safe to do it. The replica is that place, and the fidelity you give it decides which findings you are able to confirm at all.

> **At CyberTravels.** You cannot confirm the IDOR by exploiting it in production. The replica is where the booking API can be attacked safely, and its fidelity decides which findings are confirmable at all.

## 2 · The framework

```
   production            replica
   +-----------+         +------------------+
   | real data |   -->   | stubbed data     |
   | real deps |         | recorded deps    |
   | real users|         | nobody           |
   +-----------+         +------------------+
                                 |
                        exploit here, safely, on purpose

   fidelity decides which findings you can confirm at all
```

Phase 4 turns hypotheses into facts by running the application. That is only
safe if the thing you run it against cannot hurt anyone.

**Stage 11 — Sandbox replication.** Deploy the application in an isolated,
disposable runtime: its own container, its own synthetic data, no route to
production, no real credentials.

The reason this is a *stage* rather than a footnote is that the obvious shortcut
— point the dynamic tests at staging — converts every destructive probe into an
incident. Staging usually shares an identity provider, a message bus, sometimes
a database replica, and always someone's on-call rota.

Four isolation properties, and you need all four:

- **network** — no egress except to the replica itself,
- **credentials** — synthetic secrets, so a leak is worthless,
- **data** — synthetic records, so an exfiltration test exfiltrates nothing,
- **lifetime** — destroyed after the run, so state cannot leak between tests.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The stage, as a skill

Before anything is executed against CyberTravels' environment, four checks decide whether it is a replica or staging with a different DNS name. The skill runs them — egress, credentials, data, and the destructive probes you would only run somewhere built to be destroyed.

In [ ]:
# skills/appsec/exploit-replica-isolation-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: exploit-replica-isolation-check
description: >-
  Check that the environment a finding is proved in is a replica rather than
  staging — on egress, credentials, data realism and destructive probes — before
  anything is executed against it. Use before dynamic testing, exploit
  validation, or letting an agent run a proof of concept.
allowed-tools: Read, Grep, Glob, Bash
---

# Staging is production with a different DNS name

The reason exploitation is safe is the environment, and "staging" is not that
environment: it holds real credentials, real-shaped customer records, and a
route to things that matter. A replica is built to be destroyed, and the four
checks that distinguish them take minutes.

## When to use this

Before running any probe that could change state, before pointing an agent at a
target, and every time somebody offers staging because the replica is not ready.

## Procedure

**1 — Check egress.** The environment should reach its own internal hosts and
nothing else. Test the three that matter specifically: a code host, the cloud
metadata address, and any private range. A replica that can reach GitHub can
exfiltrate.

**2 — Check credentials.** Every secret in the environment should be synthetic.
Grep the environment, the mounted files and the database. One real key makes the
whole environment production for the purposes of this decision.

**3 — Check the data.** Real-shaped is not the same as real. Sample records and
confirm they are generated, not copied — a customer row with a real email
address is a breach waiting for a log line.

**4 — Run the destructive probes.** Drop a table, delete a file, exhaust a
quota. In a replica these are unremarkable. If you would not do them here, this
is not the environment to prove an exploit in, and that is the finding.

**5 — Record the verdict per check, not overall.** "Isolated: no" sends people
looking; "credentials: fail, egress: pass" tells them what to fix.

## Output contract

```json
{
  "environment": "str",
  "checks": {"egress": {"pass": false, "reachable": ["str"]},
             "credentials": {"pass": false, "real_found": ["str"]},
             "data": {"pass": false, "sample_real": ["str"]},
             "destructive": {"pass": false, "refused": ["str"]}},
  "verdict": "replica|not a replica",
  "safe_to_exploit": false
}
```

## Failure modes

- **Accepting the name.** "staging" and "replica" are labels; the four checks
  are the definition.
- **Testing general egress only.** The metadata address is the one that turns a
  probe into a credential theft.
- **Skipping the destructive probes** because they are destructive. That
  reluctance is the answer.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/appsec/exploit-replica-isolation-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/appsec/exploit-replica-isolation-check/scripts/exploit_replica_isolation_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check that the environment an exploit is proved in is a replica and not staging, on egress, credentials, data and destructive probes.

This is the executable half of the `exploit-replica-isolation-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import re
from dataclasses import dataclass, field
from urllib.parse import urlparse

PRIVATE = [re.compile(p) for p in (r"^127\.", r"^10\.", r"^169\.254\.",
                                   r"^192\.168\.", r"^localhost$")]

@dataclass
class Sandbox:
    name: str
    allow_hosts: set = field(default_factory=set)
    credentials: dict = field(default_factory=dict)
    data: dict = field(default_factory=dict)
    ephemeral: bool = True
    destroyed: bool = False
    log: list = field(default_factory=list)

    def egress(self, url):
        host = (urlparse(url).hostname or "").lower()
        if host in self.allow_hosts:
            d = (True, "replica-internal")
        elif any(p.match(host) for p in PRIVATE):
            d = (False, "private address outside the replica — blocked")
        else:
            d = (False, "not on the replica allowlist")
        self.log.append((url, d[0], d[1])); return d

    def destroy(self):
        self.credentials.clear(); self.data.clear(); self.destroyed = True
        return "replica destroyed; state cannot leak into the next run"

REPLICA = Sandbox("appsec-replica-8812",
                  allow_hosts={"replica.local", "db.replica.local"},
                  credentials={"DB_PASSWORD": "synthetic-not-real-0000",
                               "API_TOKEN": "synthetic-not-real-1111"},
                  data={"users": [{"id": 1, "name": "test-user-a",
                                   "card": "4000000000000000"}]})

for url in ["http://replica.local/reports",
            "http://db.replica.local:5432/",
            "https://api.github.com/",
            "http://169.254.169.254/latest/meta-data/",
            "http://10.0.3.14:9200/_search"]:
    ok, why = REPLICA.egress(url)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {url[:44]:46s} {why}")

STAGING = Sandbox("staging",
                  allow_hosts={"staging.internal", "db.staging.internal",
                               "sso.corp", "bus.corp", "api.github.com"},
                  credentials={"DB_PASSWORD": "real-staging-password",
                               "API_TOKEN": "ghp_real_staging_token"},
                  data={"users": [{"id": 4471, "name": "dana@corp",
                                   "card": "4111111111111111"}]},
                  ephemeral=False)

DESTRUCTIVE_PROBES = [
 ("drop a table",        "db.staging.internal", "db.replica.local"),
 ("exfiltrate user rows","api.github.com",      "replica.local"),
 ("brute-force login",   "sso.corp",            "replica.local"),
]
print(f"{'probe':24s}{'on staging':>14}{'on replica':>14}")
print("-" * 56)
for probe, staging_target, replica_target in DESTRUCTIVE_PROBES:
    s_ok, _ = STAGING.egress(f"http://{staging_target}/")
    r_ok, _ = REPLICA.egress(f"http://{replica_target}/")
    print(f"{probe:24s}{'REACHES':>14}{'contained':>14}" if s_ok
          else f"{probe:24s}{'blocked':>14}{'contained':>14}")

print("\nwhat a leaked credential is worth:")
for name, box in (("staging", STAGING), ("replica", REPLICA)):
    creds = list(box.credentials.values())
    synthetic = all("synthetic" in c for c in creds)
    print(f"   {name:10s}{'worthless — synthetic' if synthetic else 'REAL — usable against real systems'}")

print("\nwhat an exfiltrated record is worth:")
for name, box in (("staging", STAGING), ("replica", REPLICA)):
    rec = box.data["users"][0]
    real = not rec["name"].startswith("test-")
    print(f"   {name:10s}{'REAL customer data' if real else 'synthetic'}  {rec['name']}")

def isolation_report(box):
    checks = {
      "network": all(not ok for url, ok, _ in box.log
                     if urlparse(url).hostname not in box.allow_hosts),
      "credentials": all("synthetic" in v for v in box.credentials.values()) if box.credentials else True,
      "data": all(u["name"].startswith("test-") for u in box.data.get("users", [])),
      "lifetime": box.ephemeral,
    }
    return checks, all(checks.values())

for box in (REPLICA, STAGING):
    checks, ok = isolation_report(box)
    print(f"{box.name}")
    for k, v in checks.items():
        print(f"   {'PASS' if v else 'FAIL':5s} {k}")
    print(f"   → suitable for dynamic testing: {ok}\n")

_, replica_ok = isolation_report(REPLICA)
_, staging_ok = isolation_report(STAGING)
assert replica_ok and not staging_ok

print(REPLICA.destroy())
print(f"credentials after destroy: {REPLICA.credentials or 'cleared'}")
print(f"data after destroy:        {REPLICA.data or 'cleared'}")
assert REPLICA.destroyed and not REPLICA.credentials

## What you just proved

The replica permits only its own internal hosts and blocks GitHub, the metadata service and private addresses. Staging holds real credentials and a real-shaped customer record while the replica holds synthetic ones. The four isolation checks pass for the replica and fail for staging on credentials, data and lifetime, and destroying the replica clears its state.

## Your turn

Check whether your dynamic testing currently runs against staging. If it does, list what staging shares with production — identity provider, message bus, data replica. Each shared component is a path from a test probe to a real incident.

---

**Next → [B2.7 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*